# 🎙️ MenuiPro — voix en wolof avec Adia TTS

Ce cahier fabrique **tous les audios du guide vocal** de MenuiPro en une fois, avec le modèle **Adia TTS** de CONCREE (licence Apache 2.0, usage commercial autorisé en citant CONCREE : https://huggingface.co/CONCREE/Adia_TTS).

**Mode d'emploi**
1. Menu **Exécution → Modifier le type d'exécution → GPU T4** (gratuit), puis Enregistrer.
2. **Relisez les phrases wolof** de la cellule 2 et corrigez-les si besoin. Ne changez pas les clés à gauche (`bienvenue`, `choisir`…).
3. Menu **Exécution → Tout exécuter**. Comptez 10 à 15 minutes. À la fin, le fichier `voix-wolof.zip` se télécharge.
4. Décompressez-le, puis, dans MenuiPro → Compte (admin) → 🎙️ Voix en wolof, touchez **📦 Importer les audios** et sélectionnez tous les fichiers `.mp3`.

Une phrase mal prononcée ? Relancez seulement cette phrase (cellule 5), ou enregistrez-la au micro dans la console.

In [ ]:
# 1. Installation (2 à 3 minutes)
!pip -q install git+https://github.com/huggingface/parler-tts.git soundfile
!nvidia-smi -L

In [ ]:
# 2. Les phrases en wolof — À RELIRE par un locuteur natif. Gardez les clés telles quelles.
# Moins de 200 caractères par phrase (limite du modèle).
PHRASES = {
 "bienvenue": "Dalal ak jàmm ci MenuiPro. Tànnal meuble bi nga bëgg defar.",
 "choisir": "Bësal nataalu meuble bi.",
 "cotes": "Bindal yaatuwaay bi, kawewaay bi ak xóotaay bi, ci centimètre.",
 "options": "Bësal plus walla moins ngir soppi tiroir yi, bunt yi ak étagère yi.",
 "resultat": "Fii nga ciy gis ñaata feuille walla ñaata barre bant nga wara jënd.",
 "vue3d": "Bësal bunt walla tiroir ngir ubbi ko. Wëndeelal meuble bi ak sa baaraam.",
 "plan": "Lii mooy plan découpe bi. Bépp rectangle, ab pièce la boo wara dagg.",
 "achat": "Lii mooy lépp lu nga wara jënd, ak njëgam.",
 "commander": "Bësal bouton bu wert bi ngir yónne commande bi ci WhatsApp.",
 "devis": "Bindal turu client bi, te yónne ko devis bi ci nataal.",
 "debit": "Liste débit bi dafay wax taille bu pièce bu nekk.",
 "bois": "Ci bant bu dëgër, appli bi dafay xayma planche, chevron ak madrier yi nga wara jënd.",
 "cnc": "Bouton bii dafay waajal fichier yi ngir machine CNC bi.",
 "projets": "Fii nga ciy gis sa meuble yépp yoo denc.",
 "catalogue": "Fii lañuy bind njëgu panneau yi, bant yi ak accessoire yi.",
 "atelier": "Bindal sa tur, sa nimero telefon ak sa njëgu bés.",
 "compte": "Duggal ak sa nimero telefon ak sa code bu am juróom benni chiffre.",
 "code_quinc": "Bindal code bi sa quincaillerie jox la.",
 "ecoute": "Maa ngi lay déglu. Waxal meuble bi ak ay mesuram, te bësaatal micro bi.",
 "compris": "Waaw. Meuble bi a ngi.",
 "modifie": "Soppi nañu ko.",
 "pas_compris": "Dégguma. Waxaatal ndànk, bu la neexee.",
 "limite": "Wax nga lu bare tey. Jéemaatal suba.",
 "enregistre": "Denc nañu ko.",
 "erreur": "Am na jafe-jafe. Saytul te jéemaat."
}

# Style de voix (description en anglais, comme dans la fiche du modèle)
VOIX = "A clear and educational voice, with a flow adapted to learning"
for k, t in PHRASES.items():
    assert len(t) <= 200, k
print(len(PHRASES), 'phrases')

In [ ]:
# 3. Chargement du modèle Adia TTS (environ 2 Go)
import torch, soundfile as sf
from parler_tts import ParlerTTSForConditionalGeneration
from transformers import AutoTokenizer

appareil = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Appareil :', appareil, '(si « cpu », activez le GPU : étape 1 du mode d\'emploi)')
modele = ParlerTTSForConditionalGeneration.from_pretrained('CONCREE/Adia_TTS').to(appareil)
tokenizer = AutoTokenizer.from_pretrained('CONCREE/Adia_TTS')

def generer(texte, fichier):
    ids_voix = tokenizer(VOIX, return_tensors='pt').input_ids.to(appareil)
    ids_texte = tokenizer(texte, return_tensors='pt').input_ids.to(appareil)
    audio = modele.generate(input_ids=ids_voix, prompt_input_ids=ids_texte,
                            temperature=0.8, max_new_tokens=1000, do_sample=True, top_k=50, repetition_penalty=1.2)
    sf.write(fichier, audio.cpu().numpy().squeeze(), modele.config.sampling_rate)

In [ ]:
# 4. Génération de toutes les phrases → mp3 (≈ 20 s par phrase)
import os, subprocess, time
os.makedirs('voix', exist_ok=True)
for i, (cle, texte) in enumerate(PHRASES.items(), 1):
    t = time.time()
    generer(texte, f'voix/{cle}.wav')
    subprocess.run(['ffmpeg', '-y', '-loglevel', 'error', '-i', f'voix/{cle}.wav', '-ac', '1', '-b:a', '64k', f'voix/{cle}.mp3'], check=True)
    os.remove(f'voix/{cle}.wav')
    print(f'{i}/{len(PHRASES)} {cle} ✅ ({time.time()-t:.0f} s)')

In [ ]:
# 5. (Facultatif) Écouter, et refaire une phrase mal prononcée
from IPython.display import Audio, display
for cle in PHRASES:
    print(cle, '—', PHRASES[cle]); display(Audio(f'voix/{cle}.mp3'))

# Pour refaire une seule phrase, enlevez le # devant les 2 lignes suivantes et mettez sa clé :
# cle = 'bienvenue'
# generer(PHRASES[cle], f'voix/{cle}.wav'); subprocess.run(['ffmpeg','-y','-loglevel','error','-i',f'voix/{cle}.wav','-ac','1','-b:a','64k',f'voix/{cle}.mp3'], check=True); display(Audio(f'voix/{cle}.mp3'))

In [ ]:
# 6. Téléchargement du zip
import shutil
shutil.make_archive('voix-wolof', 'zip', 'voix')
from google.colab import files
files.download('voix-wolof.zip')